# Phase 1 · Lesson 5: Git & GitHub, How Version Control Really Works
**ML Engineer Journey · Benson M. Merita**

This lesson has **two parts**:
1. **This notebook:** what Git *is*, mathematically: content-addressed hashing and a directed acyclic graph. You'll build a mini-Git in Python whose object IDs match real Git's byte for byte.
2. **`GIT_LAB.md`:** the hands-on part in **Git Bash**. You turn your `MachineLearning` folder into a GitHub repository and practise the branch → commit → push → pull-request workflow that ML teams use every day.

Recruiters look at your GitHub before your CV. This lesson is what makes that profile real.

| Section | What you get |
|---|---|
| 1 · Intuition | Save points, parallel drafts, a shared copy |
| 2 · The math | SHA-1 content addressing, Merkle DAG, merge base, diff as LCS |
| 3 · From scratch | Git-exact blob/tree/commit hashing, branches, merge base, tamper detection, LCS diff |
| 4 · With the real tool | `git` via subprocess: verify your hashes, branch, merge, resolve a conflict |
| 5 · Dataset | Your own repository (and its history as data) |
| 6–8 | Exercises, quiz, mistakes & interview questions |
| 🎬 Demo | Animated commit graph |

---
## 1 · Intuition

**Save points.** In a video game you save before the boss fight so you can go back if it goes wrong. A **commit** is a save point for your whole project folder, with a note explaining what changed.

**Parallel drafts.** Want to try a new feature-engineering idea without breaking the working notebook? Make a **branch**, a parallel line of work. If the idea works, **merge** it back. If not, delete the branch, and nothing is harmed.

**A shared copy.** **GitHub** hosts a copy of your repository online. It's your backup, your collaboration space (pull requests, reviews), and, for you right now, your **public portfolio**.

**Real ML engineering connection:** every ML team works through Git and pull requests. The same ideas, hashing content to identify versions and keeping a graph of history, power the **data and model versioning** tools you'll meet in Phase 7 (DVC, MLflow model registry). A model in production must be traceable to the exact code commit and data version that produced it.

---
## 2 · The math

**Content addressing.** Git never names a file version by its filename. It names it by a **hash of its contents**:
$$\text{id}(\text{blob}) = \operatorname{SHA1}\big(\texttt{"blob "} \,\|\, \text{len} \,\|\, \texttt{\\0} \,\|\, \text{content}\big)$$
SHA-1 maps any input to a 160-bit number, so equal content gives an equal ID and storage is deduplicated automatically.

**A Merkle structure.** A **tree** (a folder snapshot) is hashed from its entries' names and child hashes. A **commit** is hashed from its tree hash, its **parent commit hash(es)**, author, time and message:
$$\text{id}(c_k) = \operatorname{SHA1}\big(\text{tree}_k,\ \text{id}(c_{k-1}),\ \text{meta}_k\big)$$
So every commit ID depends on **the entire history before it**. Change one byte in an old file and every later commit ID changes, which makes history tamper-evident. Blockchains use the same idea.

**History is a DAG.** Commits are nodes, and edges point to parents. A normal commit has 1 parent, a merge commit has 2, and the first commit has 0. There are no cycles, because a commit can't be its own ancestor: its hash would have to contain itself.
**Branches are just pointers**: a branch is a name mapping to one commit ID. Creating a branch costs 41 bytes.

**Merge base.** To merge branch $B$ into $A$, Git finds the **best common ancestor**: the latest commit in $\text{anc}(A) \cap \text{anc}(B)$. It then compares base→A and base→B (a **three-way merge**). If both sides changed the same lines differently, that's a **conflict** you resolve by hand.

**Diff = longest common subsequence.** To show what changed between two versions of a file (sequences of lines $a$, $b$), find the LCS by dynamic programming:
$$L[i,j] = \begin{cases} L[i-1,j-1] + 1 & a_i = b_j \\ \max(L[i-1,j],\, L[i,j-1]) & \text{otherwise}\end{cases}$$
Lines not in the LCS are deletions (`-`) or insertions (`+`). Git uses Myers' algorithm, a faster route to the same shortest edit script.

**Collisions?** By the birthday bound, with $n$ objects $P(\text{collision}) \approx n^2 / 2^{161}$. With a billion objects that's about $10^{-31}$. (Git is migrating to SHA-256 because SHA-1 has known *deliberate* attacks, not accidental collisions.)

---
## 3 · From scratch: a mini-Git with real Git IDs
### 3.1 Blobs: hashing file contents exactly like Git

In [1]:
import hashlib, time
import numpy as np

def hash_object(kind: str, data: bytes) -> str:
    """Git's object ID: sha1( b"<kind> <len>\0" + data )."""
    header = f"{kind} {len(data)}".encode() + b"\x00"
    return hashlib.sha1(header + data).hexdigest()

blob_a = hash_object("blob", b"hello ML\n")
blob_b = hash_object("blob", b"hello ML!\n")       # one character different
print("blob('hello ML')  :", blob_a)
print("blob('hello ML!') :", blob_b)
same = sum(x == y for x, y in zip(blob_a, blob_b))
print(f"hex digits in common: {same}/40  <- a tiny change scrambles everything (avalanche effect)")

blob('hello ML')  : edf0662362c0e815acae59fc8ced72542b46be94
blob('hello ML!') : 74623713f94e5f8672c681d2227bf34d4aae71f6
hex digits in common: 1/40  <- a tiny change scrambles everything (avalanche effect)


*Real Git gives `edf0662…` for `hello ML\n`. Section 4 checks this against the real `git hash-object`.*

### 3.2 Trees and commits: the Merkle chain

In [2]:
def hash_tree(files: dict) -> tuple[str, dict]:
    """files: {name: bytes}. A flat tree, exactly as Git stores it: sorted entries of
    b"100644 <name>\0" + 20 raw bytes of the blob id."""
    blobs = {name: hash_object("blob", data) for name, data in files.items()}
    body = b"".join(f"100644 {name}".encode() + b"\x00" + bytes.fromhex(blobs[name])
                    for name in sorted(files))
    return hash_object("tree", body), blobs

def hash_commit(tree: str, parents: list, author: str, timestamp: int, message: str) -> str:
    lines = [f"tree {tree}"] + [f"parent {p}" for p in parents]
    lines += [f"author {author} {timestamp} +0000", f"committer {author} {timestamp} +0000", "", message]
    return hash_object("commit", ("\n".join(lines) + "\n").encode())

AUTHOR = "Benson M. Merita <benson@example.com>"
T0 = 1_794_000_000                                   # fixed timestamps -> reproducible IDs

### 3.3 `MiniGit`: commits, branches, log, merge base

In [3]:
class MiniGit:
    def __init__(self):
        self.commits = {}            # id -> {"parents": [...], "message": str, "files": {...}}
        self.branches = {}           # name -> commit id   (a branch is JUST a pointer)
        self.head = "main"
        self.clock = T0

    def commit(self, files: dict, message: str, parents=None) -> str:
        if parents is None:
            parents = [self.branches[self.head]] if self.head in self.branches else []
        tree, _ = hash_tree(files)
        self.clock += 60
        cid = hash_commit(tree, parents, AUTHOR, self.clock, message)
        self.commits[cid] = {"parents": parents, "message": message, "files": dict(files), "tree": tree, "time": self.clock}
        self.branches[self.head] = cid               # move the current branch pointer forward
        return cid

    def branch(self, name):  self.branches[name] = self.branches[self.head]
    def switch(self, name):  self.head = name

    def ancestors(self, cid) -> set:
        seen, stack = set(), [cid]
        while stack:                                 # depth-first walk along parent edges
            c = stack.pop()
            if c not in seen:
                seen.add(c); stack.extend(self.commits[c]["parents"])
        return seen

    def merge_base(self, a, b) -> str:
        common = self.ancestors(self.branches[a]) & self.ancestors(self.branches[b])
        return max(common, key=lambda c: self.commits[c]["time"])   # latest common ancestor (fine for simple histories)

    def log(self, name=None):
        c = self.branches[name or self.head]
        while c:
            p = self.commits[c]["parents"]
            print(c[:7], ("(merge) " if len(p) > 1 else "") + self.commits[c]["message"])
            c = p[0] if p else None                  # follow first parent, like `git log --first-parent`

In [4]:
repo = MiniGit()
c1 = repo.commit({"README.md": b"# ML journey\n"}, "Add README")
c2 = repo.commit({"README.md": b"# ML journey\n", "lesson01.ipynb": b"{numpy}\n"}, "Lesson 1: NumPy")

repo.branch("lesson02"); repo.switch("lesson02")
c3 = repo.commit({"README.md": b"# ML journey\n", "lesson01.ipynb": b"{numpy}\n", "lesson02.ipynb": b"{pca}\n"}, "Lesson 2: PCA")

repo.switch("main")
c4 = repo.commit({"README.md": b"# ML journey\nRoadmap\n", "lesson01.ipynb": b"{numpy}\n"}, "Update roadmap")

base = repo.merge_base("main", "lesson02")
print("merge base is c2 ('Lesson 1')?", base == c2)
merged_files = {"README.md": b"# ML journey\nRoadmap\n", "lesson01.ipynb": b"{numpy}\n", "lesson02.ipynb": b"{pca}\n"}
c5 = repo.commit(merged_files, "Merge branch 'lesson02'", parents=[repo.branches["main"], repo.branches["lesson02"]])
print("branches:", {k: v[:7] for k, v in repo.branches.items()})
print()
repo.log()

merge base is c2 ('Lesson 1')? True
branches: {'main': 'a76ec3e', 'lesson02': '0f45fc2'}

a76ec3e (merge) Merge branch 'lesson02'
a541fd3 Update roadmap
ee08ddb Lesson 1: NumPy
7b6616b Add README


*`main` and `lesson02` are 40-character pointers into one shared graph. The merge commit has two parents. And a merge base is just set intersection plus "latest".*

### 3.4 Tamper detection: why history can be trusted

In [5]:
def verify(repo, cid):
    """Recompute a commit's ID from its stored content and its (recursively verified) parents."""
    c = repo.commits[cid]
    for p in c["parents"]:
        verify(repo, p)
    tree, _ = hash_tree(c["files"])
    recomputed = hash_commit(tree, c["parents"], AUTHOR, c["time"], c["message"])
    if recomputed != cid:
        raise ValueError(f"corrupt history: commit {cid[:7]} ('{c['message']}') does not match its content")
    return True

print("history valid:", verify(repo, repo.branches["main"]))

history valid: True


**Intentional error: secretly edit an old file and re-verify:**

In [6]:
repo.commits[c2]["files"]["lesson01.ipynb"] = b"{numpy, but edited later}\n"   # rewrite the past
verify(repo, repo.branches["main"])

ValueError: corrupt history: commit ee08ddb ('Lesson 1: NumPy') does not match its content

*`ValueError: corrupt history`. The stored ID no longer matches the content. In real Git you can't edit a commit in place. Tools that "change history" (`commit --amend`, `rebase`) create **new** commits with new IDs. That's why rewriting history that others already pulled causes trouble.*

In [7]:
repo.commits[c2]["files"]["lesson01.ipynb"] = b"{numpy}\n"   # restore
print("restored, valid again:", verify(repo, repo.branches["main"]))

restored, valid again: True


### 3.5 `diff` from scratch: LCS by dynamic programming (NumPy table)

In [8]:
def lcs_diff(a: list, b: list) -> list:
    n, m = len(a), len(b)
    L = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(n - 1, -1, -1):                     # fill the table backwards: L[i,j] = LCS of a[i:], b[j:]
        for j in range(m - 1, -1, -1):
            L[i, j] = L[i + 1, j + 1] + 1 if a[i] == b[j] else max(L[i + 1, j], L[i, j + 1])
    out, i, j = [], 0, 0
    while i < n and j < m:                             # walk the table to recover the edit script
        if a[i] == b[j]:          out.append("  " + a[i]); i += 1; j += 1
        elif L[i + 1, j] >= L[i, j + 1]: out.append("- " + a[i]); i += 1
        else:                     out.append("+ " + b[j]); j += 1
    out += ["- " + x for x in a[i:]] + ["+ " + x for x in b[j:]]
    return out

old = ["import numpy as np", "X = load()", "w = np.linalg.inv(X.T @ X) @ X.T @ y", "print(w)"]
new = ["import numpy as np", "X = load()", "w = np.linalg.solve(X.T @ X, X.T @ y)", "print(w.round(3))"]
print("\n".join(lcs_diff(old, new)))

  import numpy as np
  X = load()
- w = np.linalg.inv(X.T @ X) @ X.T @ y
- print(w)
+ w = np.linalg.solve(X.T @ X, X.T @ y)
+ print(w.round(3))


*`-` lines were removed and `+` lines added. This is what you read in every pull request. (The example also happens to be the `inv` → `solve` fix from Lesson 2.)*

---
## 4 · With the real tool: `git` from Python
These cells call the real `git` executable (Git for Windows puts it on your PATH) inside a throwaway temp folder, so your real repo is untouched.

In [9]:
import subprocess, tempfile, os, pathlib

work = pathlib.Path(tempfile.mkdtemp())
ENV = {**os.environ, "GIT_AUTHOR_NAME": "Benson M. Merita", "GIT_AUTHOR_EMAIL": "benson@example.com",
       "GIT_COMMITTER_NAME": "Benson M. Merita", "GIT_COMMITTER_EMAIL": "benson@example.com"}

def git(*args, check=True, env=None):
    r = subprocess.run(["git", *args], cwd=work, capture_output=True, text=True, env=env or ENV)
    if check and r.returncode != 0:
        raise RuntimeError(r.stderr)
    return (r.stdout + r.stderr).strip()

def write(name, text):
    (work / name).write_bytes(text.encode())       # write bytes: no Windows CRLF surprises

git("init", "-q", "-b", "main")
git("config", "core.autocrlf", "false")
write("a.txt", "hello ML\n")
print("real git hash-object:", git("hash-object", "a.txt"))
print("our hash_object:     ", hash_object("blob", b"hello ML\n"))

real git hash-object: edf0662362c0e815acae59fc8ced72542b46be94
our hash_object:      edf0662362c0e815acae59fc8ced72542b46be94


### 4.1 Your tree and commit IDs match real Git exactly

In [10]:
files = {"README.md": b"# ML journey\n", "a.txt": b"hello ML\n"}
for name, data in files.items():
    (work / name).write_bytes(data)
git("add", ".")
when = f"{T0} +0000"
env_fixed = {**ENV, "GIT_AUTHOR_DATE": when, "GIT_COMMITTER_DATE": when}
git("-c", "commit.gpgsign=false", "commit", "-q", "-m", "First commit", env=env_fixed)   # unsigned, so content = our formula

real_tree = git("rev-parse", "HEAD^{tree}")
real_commit = git("rev-parse", "HEAD")
our_tree, _ = hash_tree(files)
our_commit = hash_commit(our_tree, [], AUTHOR, T0, "First commit")
print("tree   real:", real_tree, "| ours:", our_tree, "| match:", real_tree == our_tree)
print("commit real:", real_commit, "| ours:", our_commit, "| match:", real_commit == our_commit)

tree   real: 675a751be654136f0019460bf022f57bba624dad | ours: 675a751be654136f0019460bf022f57bba624dad | match: True
commit real: 67de3f6e82bbbff39ae498c8a3473af0f6f6731b | ours: 67de3f6e82bbbff39ae498c8a3473af0f6f6731b | match: True


*Byte-for-byte identical. You now know exactly what a commit ID is: a hash of a snapshot, its parents, the author, the time and the message. (If commit **signing** is on, the signature becomes part of the commit object too. That's why the cell turns it off for this comparison. Signed commits show a Verified badge on GitHub.)*

### 4.2 Branch, diverge, merge, and a real conflict

In [11]:
git("switch", "-q", "-c", "feature-scaling")
write("model.py", "scale = 'standard'\n")
git("add", "."); git("commit", "-q", "-m", "Use standard scaling")

git("switch", "-q", "main")
write("model.py", "scale = 'minmax'\n")
git("add", "."); git("commit", "-q", "-m", "Use min-max scaling")

out = git("merge", "feature-scaling", check=False)
print(out)
print("\n--- model.py now contains conflict markers ---")
print((work / "model.py").read_text())

Auto-merging model.py
CONFLICT (add/add): Merge conflict in model.py
Automatic merge failed; fix conflicts and then commit the result.

--- model.py now contains conflict markers ---
<<<<<<< HEAD
scale = 'minmax'
scale = 'standard'
>>>>>>> feature-scaling



*Both branches changed the **same line** differently, so Git stops and asks you to decide. Between `<<<<<<<` and `=======` is your current branch (HEAD), and between `=======` and `>>>>>>>` is the incoming branch. You edit the file to the final version, delete the markers, then `add` and `commit`.*

In [12]:
write("model.py", "scale = 'standard'   # chosen after comparing CV scores\n")
git("add", "model.py")
git("commit", "-q", "-m", "Merge feature-scaling: keep standard scaling")
print(git("log", "--oneline", "--graph", "--all"))
print("\nmerge commit parents:", len(git("log", "-1", "--format=%P").split()))

*   32745ff Merge feature-scaling: keep standard scaling
|\  
| * 7514e13 Use standard scaling
* | 585bc41 Use min-max scaling
|/  
* 67de3f6 First commit

merge commit parents: 2


*`git log --graph` draws the same DAG you built in `MiniGit`, and the merge commit has 2 parents.*

| MiniGit (yours) | Real Git |
|---|---|
| `hash_object("blob", data)` | `git hash-object file` |
| `hash_tree`, `hash_commit` | `git add` + `git commit` (`git write-tree`, `git commit-tree` underneath) |
| `branches[name] = id` | `git branch name` / `git switch -c name` |
| `merge_base(a, b)` | `git merge-base a b` |
| `lcs_diff(old, new)` | `git diff` |
| `verify(...)` | `git fsck` |

---
## 5 · Practice dataset: your own repository
The dataset for this lesson is **your `MachineLearning` repository** and its history. Every commit is a row: hash, author, date, message, and files changed. Exercise 5 turns `git log` into a Pandas DataFrame, the same kind of analysis teams run on engineering activity.

Follow **`GIT_LAB.md`** now to put your repository on GitHub, then come back to the exercises.

---
## 🎬 Demo: the animated commit graph (`assets/commit_graph.gif`)

In [13]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# (x, lane, label, parents)  lane 0 = main, lane 1 = branch
nodes = [(0, 0, "README", []), (1, 0, "Lesson 1", [0]), (2, 1, "Lesson 2", [1]), (3, 1, "PCA demo", [2]),
         (3, 0, "Roadmap", [1]), (4, 0, "Merge", [4, 3]), (5, 0, "Lesson 3", [5])]
colors = {0: "#264653", 1: "#e76f51"}
fig, ax = plt.subplots(figsize=(8, 3), dpi=110)

def update(k):
    ax.clear(); ax.set_xlim(-0.6, 5.8); ax.set_ylim(-0.8, 1.8); ax.axis("off")
    shown = nodes[: k + 1]
    for i, (x, lane, lab, parents) in enumerate(shown):
        for p in parents:
            px, pl = nodes[p][0], nodes[p][1]
            ax.annotate("", xy=(px, pl), xytext=(x, lane),
                        arrowprops=dict(arrowstyle="->", color="grey", lw=1.5, shrinkA=12, shrinkB=12))
        ax.scatter([x], [lane], s=420, color=colors[lane], zorder=3)
        ax.text(x, lane + (0.42 if lane else -0.5), lab, ha="center", fontsize=9)
    main_tip = max([i for i, n in enumerate(shown) if n[1] == 0])
    ax.text(shown[main_tip][0], -0.75, "main", ha="center", color=colors[0], fontweight="bold")
    br = [i for i, n in enumerate(shown) if n[1] == 1]
    if br and k < 5:
        ax.text(shown[br[-1]][0], 1.7, "lesson02", ha="center", color=colors[1], fontweight="bold")
    msg = "merge commit = 2 parents" if k == 5 else "arrows point to parents; branches are pointers"
    ax.set_title(f"Git history is a DAG: {msg}", loc="left", fontsize=10, fontweight="bold")

frames = list(range(len(nodes))) + [len(nodes) - 1] * 8      # hold the final frame
anim = FuncAnimation(fig, update, frames=frames)
anim.save("assets/commit_graph.gif", writer=PillowWriter(fps=1.5))
update(len(nodes) - 1); fig.savefig("assets/commit_graph.png", bbox_inches="tight")
plt.close(fig)
print("Saved assets/commit_graph.gif and .png")

Saved assets/commit_graph.gif and .png


---
## 6 · Exercises (answers hidden; send your repo link and code when ready)
1. **(Easy, practical) Ship it:** Complete `GIT_LAB.md` Parts 1–3 so all lessons are on GitHub. Send me the repo URL.
2. **(Easy, practical) Branch + PR:** Do Part 4: put your Lesson 1–4 exercise notebooks on a branch called `exercises`, open a pull request, and merge it on GitHub. Send me the PR link. (This also gets me your exercises to review.)
3. **(Medium) Conflict drill:** Follow Part 5 to create and resolve a merge conflict *in your real repo*. Paste the conflict markers you saw and your final version.
4. **(Medium–Hard) Nested trees:** Extend `hash_tree` to handle folders (a folder entry uses mode `40000` and the *tree* ID of the sub-folder). Verify your result against `git rev-parse HEAD^{tree}` for a temp repo with `data/x.csv` and `README.md`. *(Hint: Git sorts entries by name, treating a folder name as if it ended with `/`.)*
5. **(Challenge) History as data:** Run `git log --pretty=format:"%h|%an|%ad|%s" --date=iso --numstat` on your repo, parse it into a DataFrame (one row per commit: hash, date, message, lines added, lines deleted), and chart commits per day plus lines changed per lesson. You'll have a commit-activity chart to post.

## 7 · Mini-quiz
1. Why does changing one byte in an old file change the ID of every later commit?
2. What *exactly* is a branch in Git, and why is creating one almost free?
3. How many parents does a merge commit have? The very first commit?
4. A teammate ran `git push --force` on `main` after rebasing. Why might your next `git pull` go badly?

## 8 · Common mistakes & interview questions
**Common mistakes**
- **Committing data, models or secrets.** Large CSVs, `.pkl` models and API keys don't belong in Git. Use `.gitignore`, and keep keys in a `.env` file (already ignored in your repo).
- **Giant "update" commits.** Make one logical change per commit with a message that says *what* and *why* ("Lesson 2: add PCA via SVD exercise", not "stuff").
- **Working directly on `main`** for everything. Use a branch per lesson or feature.
- **Force-pushing shared branches**, or rewriting history others already have.
- **Ignoring Windows line endings.** Set `core.autocrlf true` on Windows so diffs don't show every line as changed.
- **Notebook diffs that are huge** because of outputs. Keep outputs for your portfolio, but clear accidental giant outputs before committing. `nbdime` gives readable notebook diffs.
- **Panicking at a conflict.** It's just markers in a text file. Read both sides, edit, `add`, `commit`.

**Interview questions**
1. Explain what happens internally when you run `git commit`.
2. `merge` vs `rebase`: what's the difference, and when would you use each?
3. How would you undo a commit that's already pushed to a shared branch? (`git revert`, not `reset`.)
4. How do you version **data and models**, not just code? (Preview: DVC, MLflow, Git LFS.)
5. Describe your team workflow: branches, pull requests, code review, CI.
6. What is a merge conflict, and how do you resolve one?

---
### ✅ Lesson 5 checklist
- [ ] Notebook runs top to bottom (it needs `git` on PATH)
- [ ] `GIT_LAB.md` Parts 1–5 done; repo is public on GitHub
- [ ] Exercises 1–5 attempted
- [ ] Quiz answered
- [ ] Post: commit-graph GIF + "my ML journey is now on GitHub" (see `SOCIAL_PACK.md`)

**Next lesson:** Writing reusable Python for ML: functions, modules, classes and the scikit-learn estimator API (`fit` / `predict` / `transform`), turning your scratch code into a small package.